In [1]:
import nflreadpy as nfl
import pandas as pd

In [2]:
SEASON = 2025
 
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [ ]:
# Load and filter to real 4th-down decisions, Q4 only 
pbp = nfl.load_pbp(seasons=[SEASON]).to_pandas()
 
fourth = pbp[
    (pbp["down"] == 4)
    & (pbp["qtr"] == 4)
    & (pbp["play_type"].isin(["run", "pass", "punt", "field_goal"]))
    & (pbp["qb_kneel"].fillna(0) != 1)
    & (pbp["qb_spike"].fillna(0) != 1)
].copy()

In [4]:
fourth["decision"] = fourth["play_type"].map(
    {"run": "go", "pass": "go", "punt": "punt", "field_goal": "field_goal"}
)
fourth["is_go"] = (fourth["decision"] == "go").astype(int)

In [5]:
# Time remaining in Q4, in minutes (quarter_seconds_remaining counts down from 900)
fourth["time_bucket"] = pd.cut(
    fourth["quarter_seconds_remaining"] / 60,
    bins=[0, 2, 5, 15],
    labels=["2-0 min", "5-2 min", "15-5 min"],
    include_lowest=True,
)

In [6]:
# cut() with increasing bins labels ascending; reorder for readability
time_order = ["15-5 min", "5-2 min", "2-0 min"]
fourth["time_bucket"] = pd.Categorical(
    fourth["time_bucket"], categories=time_order, ordered=True
)

In [7]:
# Score differential, from the possessing team's perspective, trailing = negative.
# Bucketed on absolute size: within one score (0-8) vs. more than one score (8+).
fourth["score_bucket"] = pd.cut(
    fourth["score_differential"].abs(),
    bins=[-0.01, 8, float("inf")],
    labels=["0-8", "8+"],
)

In [8]:
# Field position: yardline_100 = distance from the END ZONE, so 100 is own goal
# line and 1 is the opponent's. 100-60 = own territory, 59-40 = midfield, 39- = scoring range.
fourth["field_bucket"] = pd.cut(
    fourth["yardline_100"],
    bins=[0, 39, 59, 100],
    labels=["39 and lower (scoring range)", "59-40 (midfield)", "100-60 (own territory)"],
)
field_order = ["100-60 (own territory)", "59-40 (midfield)", "39 and lower (scoring range)"]
fourth["field_bucket"] = pd.Categorical(
    fourth["field_bucket"], categories=field_order, ordered=True
)

In [9]:
def go_rate_table(df: pd.DataFrame, index) -> pd.DataFrame:
    """Go-for-it rate (%), decision mix (%), and sample size for a grouping."""
    rate = (df.groupby(index, observed=False)["is_go"].mean() * 100).round(1)
    n = df.groupby(index, observed=False)["is_go"].count()
    mix = (
        pd.crosstab(
            [df[c] for c in index] if isinstance(index, list) else df[index],
            df["decision"],
            normalize="index",
        ) * 100
    ).round(1)
    out = mix.copy()
    out["go_rate_%"] = rate
    out["n"] = n
    return out

In [ ]:
# 1. Q4 by time remaining 
print(f"\n=== {SEASON} Q4: decisions by time remaining ===")
print(go_rate_table(fourth, "time_bucket"))


=== 2025 Q4: decisions by time remaining ===
decision     field_goal    go  punt  go_rate_%    n
time_bucket                                        
15-5 min           22.1  27.2  50.7       27.2  655
5-2 min            21.8  42.5  35.7       42.5  266
2-0 min            20.8  52.8  26.4       52.8  250


In [ ]:
# 2. Q4 by score differential
print(f"\n=== {SEASON} Q4: decisions by score differential (abs) ===")
print(go_rate_table(fourth, "score_bucket"))


=== 2025 Q4: decisions by score differential (abs) ===
decision      field_goal    go  punt  go_rate_%    n
score_bucket                                        
0-8                 25.3  29.2  45.6       29.2  586
8+                  18.3  43.1  38.6       43.1  585


In [ ]:
# 3. Q4 by field position
print(f"\n=== {SEASON} Q4: decisions by field position ===")
print(go_rate_table(fourth, "field_bucket"))


=== 2025 Q4: decisions by field position ===
decision                      field_goal    go  punt  go_rate_%    n
field_bucket                                                        
100-60 (own territory)               0.0  19.4  80.6       19.4  396
59-40 (midfield)                     3.3  44.0  52.8       44.0  307
39 and lower (scoring range)        52.4  45.1   2.6       45.1  468


In [ ]:
# 4. Combined: time x score x field position
print(f"\n=== {SEASON} Q4: combined time x score x field position ===")
combined = go_rate_table(fourth, ["time_bucket", "score_bucket", "field_bucket"])
print(combined)


=== 2025 Q4: combined time x score x field position ===
decision                                               field_goal    go  punt  go_rate_%    n
time_bucket score_bucket field_bucket                                                        
15-5 min    0-8          100-60 (own territory)               0.0   2.9  97.1        2.9  104
                         59-40 (midfield)                     3.2  23.2  73.7       23.2   95
                         39 and lower (scoring range)        68.4  29.1   2.6       29.1  117
            8+           100-60 (own territory)               0.0  13.1  86.9       13.1  122
                         59-40 (midfield)                     2.2  44.9  52.8       44.9   89
                         39 and lower (scoring range)        46.9  49.2   3.9       49.2  128
5-2 min     0-8          100-60 (own territory)               0.0  15.2  84.8       15.2   46
                         59-40 (midfield)                     3.6  42.9  53.6       42.9   28
   

In [ ]:
# combined grid
print(f"\n=== {SEASON} Q4: go-for-it rate (%) and sample size, combined grid ===")
print(combined[["go_rate_%", "n"]])


=== 2025 Q4: go-for-it rate (%) and sample size, combined grid ===
decision                                               go_rate_%    n
time_bucket score_bucket field_bucket                                
15-5 min    0-8          100-60 (own territory)              2.9  104
                         59-40 (midfield)                   23.2   95
                         39 and lower (scoring range)       29.1  117
            8+           100-60 (own territory)             13.1  122
                         59-40 (midfield)                   44.9   89
                         39 and lower (scoring range)       49.2  128
5-2 min     0-8          100-60 (own territory)             15.2   46
                         59-40 (midfield)                   42.9   28
                         39 and lower (scoring range)       45.7   46
            8+           100-60 (own territory)             34.7   49
                         59-40 (midfield)                   71.0   31
                      